# Stop delay dynamics: EDA and segment 4 modelling

Study delay propagation on Route `97-KEL` (Highway 97), initially **incoming segment 4, direction 0**. Fit a same-journey equation combining preceding-stop arrival delay with deviations in departure-to-arrival traversal time and upstream-stop dwell. Here, `t-1` means the preceding stop visit of that journey, not a previous bus or a fixed time-grid step.

Run **Preprocessing** first: imports, controls, offline cache loading and shared selected-segment observations. **EDA** then maps segments, compares delay/traversal/dwell, and follows selected dated journeys across their scheduled stops. **Modelling** at the bottom performs its own eligibility filtering, chronological split, training-only hourly baselines, coefficient fitting and held-out evaluation. The modelling section needs only Preprocessing, not the EDA plots or journey selection.

Use the `transit_env` kernel. This notebook loads a trusted local cache offline; there are no data downloads, database calls, or cache writes. Figures are exported as HTML under `notebooks/visuals/dynamics_model_notebook/`, not rendered inline, to reduce notebook and WSL memory use; fitted coefficients, baselines and evaluation tables are also saved there as CSV. The route map may fetch OpenStreetMap base-map tiles when its exported HTML is opened. Completed durations and final reported delays are retrospective feed-estimate proxies: this is not a pre-arrival forecast or proof of traffic causality.

## Preprocessing

Shared setup for EDA and modelling. Model-specific filtering, splits and baselines are intentionally deferred to the final section.

In [2]:
from math import cos, log2, radians
from pathlib import Path
import sys
from time import perf_counter

repo_root = next(
    (p for p in (Path.cwd().resolve(), Path.cwd().resolve().parent)
     if (p / 'forecasting' / 'forecasting').is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
sys.path.insert(0, str(repo_root / 'forecasting'))

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from forecasting import (
    build_route_segments, get_backtest_dataset, prepare_stop_dynamics_eda,
    make_trip_dynamics_figure, prepare_trip_dynamics_eda, select_trip_dynamics_examples,
    make_segment_timing_figure, prepare_segment_timing_eda,
    fit_stop_dynamics_model,
    make_stop_dynamics_model_figure,
)

### Select the dataset and incoming segment

The default is **segment 4, direction 0**, Route `97-KEL`, in the two-month snapshot: **Hwy 97 at Westside Rd Stn (103745) -> Queensway Exchange Bay B (102853)**, shape `51737`. Segment IDs follow the existing congestion engine for the same dataset, route, and single-direction selection; a different snapshot or direction can give the same ID a different meaning. Check the resolved metadata below.

The notebook imports the checked-out forecasting package so it can run from either the repository root or this directory. Cache loading, data preparation, and HTML export are separate: open saved HTML to zoom without rerunning the notebook. EDA baselines use the complete selected snapshot even when the display window is narrowed; modelling baselines use training visits only.

All exports belong to this notebook's folder, `notebooks/visuals/dynamics_model_notebook/`, resolved from the repository root regardless of the working directory. Plot cells write full HTML with Plotly.js embedded and print the saved path instead of displaying a figure inline; rerunning a cell overwrites its corresponding export, including modelling CSV tables. Journey exports are numbered in the current selection order, with the full dated journey identity in each figure's title. Existing files are not deleted when the selection shrinks. The route map still needs network access for its OpenStreetMap base-map tiles. Tables and lightweight diagnostics remain in the notebook.

In [3]:
backtest_dataset_id = '2026-11-10_2_months'
route_id = '97-KEL'
trip_direction_id = 0
segment_id = 4
time_bin_size = '5min'
display_timezone = 'America/Vancouver'

visuals_dir = repo_root / 'notebooks' / 'visuals' / 'dynamics_model_notebook'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'HTML plot exports: {visuals_dir}')

HTML plot exports: /home/abdul/Projects/transit-forecasting/notebooks/visuals/dynamics_model_notebook


In [4]:
load_started = perf_counter()
dataset = get_backtest_dataset(
    repo_root / 'data', source='cache', dataset_id=backtest_dataset_id,
)
static_dir = dataset.static_data_dir
stops = pd.read_csv(static_dir / 'stops.txt', usecols=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'], dtype={'stop_id': 'string'})
stop_times = pd.read_csv(static_dir / 'stop_times.txt', usecols=['trip_id', 'stop_id', 'stop_sequence'], dtype={'trip_id': 'string', 'stop_id': 'string'})
trips = pd.read_csv(static_dir / 'trips.txt', usecols=['route_id', 'trip_id', 'direction_id', 'shape_id'], dtype={'route_id': 'string', 'trip_id': 'string', 'shape_id': 'string'})
load_seconds = perf_counter() - load_started
print(f'Dataset: {dataset.directory.name}; load and static read: {load_seconds:.2f} s')
print(f'Raw vehicle / trip rows: {len(dataset.vehicle_updates):,} / {len(dataset.trip_updates):,}')

Dataset: 2026-11-10_2_months; load and static read: 4.26 s
Raw vehicle / trip rows: 685,808 / 2,194,715


### Prepare shared observations for the selected segment

`prepare_stop_dynamics_eda` builds a lightweight static catalogue with engine-compatible numbering, without recomputing route geometry. It selects latest stop predictions for exact matched endpoint visits. `prepare_segment_timing_eda` separates departure-to-arrival traversal from upstream dwell. Both EDA and modelling reuse these individual observations; speed probes are matched only for EDA comparisons. No model is trained on the time-bin medians or full-snapshot duration baselines.

Delay is the **latest reported arrival_delay**, selected by publication time, with database ID breaking ties. It is a proxy, not an independently measured arrival. Missing latest estimates are not replaced with older predictions or departure delays. Trip-start descriptors distinguish daily services; stop sequences distinguish repeated visits. Missing starts, unmatched GTFS visits, skipped/no-data/cancelled visits, unfinished arrivals, and unusable pairs are audited rather than guessed.

In [5]:
prepare_started = perf_counter()
eda = prepare_stop_dynamics_eda(
    dataset.vehicle_updates, dataset.trip_updates, stops, stop_times, trips,
    route_id=route_id, trip_direction_id=trip_direction_id,
    segment_id=segment_id, time_bin_size=time_bin_size,
)
segment_timing = prepare_segment_timing_eda(eda)
traversal_visits = segment_timing.observations.query("traversal_exclusion_reason == 'included'")
prepare_seconds = perf_counter() - prepare_started
current_visits = eda.stop_visits.query("exclusion_reason == 'included'")
upstream_visits = eda.upstream_visits.query("exclusion_reason == 'included'")
paired_visits = eda.delay_changes.query("exclusion_reason == 'included'")
display(eda.segment.to_frame('selected segment'))
print(f'Selected-segment preparation: {prepare_seconds:.2f} s')
print(f'Valid probes: {eda.speed_bins.probe_count.sum():,}; current visits: {len(current_visits):,}; upstream visits: {len(upstream_visits):,}; paired visits: {len(paired_visits):,}')
print(f'Usable traversals: {len(traversal_visits):,}; snapshot median: {segment_timing.bins.baseline_traversal_seconds.iloc[0]:.1f} s')

/tmp/ipykernel_75840/2964470504.py:2: UserWarning: Some observations were excluded or delay rows have no usable observations; inspect audit and daily coverage. Final reported delays are proxies.
  eda = prepare_stop_dynamics_eda(
/tmp/ipykernel_75840/2964470504.py:7: UserWarning: Some segment timings were excluded or have no usable observations; inspect timing audit. Durations are final feed-estimate proxies.
  segment_timing = prepare_segment_timing_eda(eda)


,selected segment
trip_direction_id,0
shape_id,51737
from_stop_id,103745
to_stop_id,102853
from_stop_sequence,5.0
to_stop_sequence,6
segment_id,4
from_stop_name,Hwy 97 at Westside Rd Stn
to_stop_name,Queensway Exchange Bay B


Selected-segment preparation: 2.90 s
Valid probes: 73,646; current visits: 2,199; upstream visits: 2,198; paired visits: 2,195
Usable traversals: 2,195; snapshot median: 420.0 s


## EDA

Retrospective coverage, geometry and individual-journey diagnostics. These plots retain full-snapshot reference medians; none supplies a modelling baseline.

### Selected route: segment map and distances

Inspect all directed stop-to-stop segments for `route_id` and `trip_direction_id` before the single-segment delay analysis. As in the Route 97 notebooks, segments follow the selected cache's GTFS shapes; different shape variants remain separate, even when their stop pairs overlap. IDs match the congestion engine and the EDA below for the same snapshot and direction. **Black** highlights `segment_id` in both plots; other segment colors match across the map and bars. Hover identifies the endpoints, shape, ID, and distance.

The horizontal bars are grouped by shape and ordered by downstream stop sequence, **not sorted by length**. Each y-axis label is the destination stop of an incoming segment, with its segment ID and shape to distinguish repeated stops. The origin has no incoming segment. Length is in **kilometres along the sliced GTFS shape**, summing great-circle distances between its vertices, not straight-line stop separation or a Shapely length in degrees. Overlapping variants can hide one another on the map but retain separate bars. Geometry is prepared from static data only; realtime delays and speeds are not recomputed for these plots.

In [ ]:
shapes = pd.read_csv(static_dir / 'shapes.txt', dtype={'shape_id': 'string'})
route_segments = build_route_segments(
    stops, stop_times, trips, shapes,
    route_id=route_id, trip_direction_id=trip_direction_id,
).sort_values(['shape_id', 'to_stop_sequence', 'segment_id']).reset_index(drop=True)
if not route_segments.segment_id.eq(segment_id).any():
    raise ValueError(f'Segment {segment_id} is not in the selected route/direction catalogue.')
route_segments['distance_km'] = route_segments.distance_meters / 1000
route_segments['color'] = [
    'black' if sid == segment_id else qualitative.Dark24[i % len(qualitative.Dark24)]
    for i, sid in enumerate(route_segments.segment_id)
]
route_segments['stop_label'] = [
    f'{row.segment_id}: {row.to_stop_name} ({row.to_stop_id}) [shape {row.shape_id}]'
    for row in route_segments.itertuples(index=False)
]
route_stop_ids = pd.concat([route_segments.from_stop_id, route_segments.to_stop_id]).unique()
route_stops = stops.loc[stops.stop_id.isin(route_stop_ids)]
print(f'{len(route_segments)} segments across {route_segments.shape_id.nunique()} shape(s)')

In [ ]:
segment_map = go.Figure()
for row in route_segments.sort_values('segment_id', key=lambda ids: ids.eq(segment_id)).itertuples(index=False):
    lon, lat = row.geometry.xy
    segment_map.add_trace(go.Scattermap(
        lon=list(lon), lat=list(lat), mode='lines',
        line=dict(color=row.color, width=8 if row.segment_id == segment_id else 5),
        name=f'Segment {row.segment_id}',
        hovertext=f'{row.from_stop_name} ({row.from_stop_id}) -> {row.to_stop_name} ({row.to_stop_id})<br>Segment {row.segment_id} | shape {row.shape_id}<br>Length: {row.distance_km:.3f} km',
        hoverinfo='text',
    ))
segment_map.add_trace(go.Scattermap(
    lon=route_stops.stop_lon, lat=route_stops.stop_lat, mode='markers',
    marker=dict(color='black', size=7), name='Stops',
    customdata=route_stops[['stop_name', 'stop_id']],
    hovertemplate='%{customdata[0]} (%{customdata[1]})<extra></extra>',
))
segment_bounds = pd.DataFrame(route_segments.geometry.map(lambda geometry: geometry.bounds).tolist())
west, south = segment_bounds[[0, 1]].min()
east, north = segment_bounds[[2, 3]].max()
map_center = dict(lon=(west + east) / 2, lat=(south + north) / 2)
map_extent = max(east - west, (north - south) / cos(radians(map_center['lat'])))
segment_map.update_layout(
    title=f'{route_id}, direction {trip_direction_id}: directed stop segments (selected: {segment_id})',
    map=dict(style='open-street-map', center=map_center,
             zoom=max(0, min(16, log2(360 / map_extent) - 0.5))),
    showlegend=False, height=650, margin=dict(t=70, b=20, l=20, r=20),
)
segment_map.write_html(visuals_dir / 'route_segment_map.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "route_segment_map.html"}')

In [ ]:
segment_lengths = go.Figure(go.Bar(
    x=route_segments.distance_km, y=route_segments.stop_label, orientation='h',
    marker_color=route_segments.color, text=route_segments.distance_km,
    texttemplate='%{text:.3f} km', textposition='outside', cliponaxis=False,
    customdata=route_segments[['segment_id', 'shape_id', 'from_stop_name', 'from_stop_id', 'to_stop_name', 'to_stop_id']],
    hovertemplate='Segment %{customdata[0]} | shape %{customdata[1]}<br>%{customdata[2]} (%{customdata[3]}) -> %{customdata[4]} (%{customdata[5]})<br>Length: %{x:.3f} km<extra></extra>',
))
segment_lengths.update_layout(
    title=f'{route_id}, direction {trip_direction_id}: incoming segment lengths',
    xaxis=dict(title='Segment length along GTFS shape (km)', rangemode='tozero'),
    yaxis=dict(title='Destination stop (incoming segment)', autorange='reversed',
               categoryorder='array', categoryarray=route_segments.stop_label.tolist(), automargin=True),
    height=max(500, 30 * len(route_segments) + 140), showlegend=False,
    margin=dict(t=80, b=70, r=100),
)
segment_lengths.write_html(visuals_dir / 'route_segment_distances.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "route_segment_distances.html"}')

### Check coverage before interpreting patterns

The two-month cache's bundled GTFS begins on November 8, so some older realtime trip IDs are unmatched. Exclusions may be uneven over time. Inspect both stage-level reasons and daily UTC coverage; do not assume every day or bus is represented.

`candidate_probes` and `candidate_updates` count selected-stop publications before exact segment matching; `matched_*` count static matches, including rows subsequently excluded for missing starts or bad measurements. Their shares measure static matching, not measurement completeness. Usable current-stop visits, preceding-stop visits, and pairs are counted by their respective reported arrival day. Raw update counts are not counts of independent buses.

In [ ]:
display(eda.audit)
display(segment_timing.audit)
coverage_table = eda.coverage.copy()
coverage_table['probe_match_share'] = coverage_table.matched_probes.div(coverage_table.candidate_probes.where(coverage_table.candidate_probes.gt(0)))
coverage_table['update_match_share'] = coverage_table.matched_updates.div(coverage_table.candidate_updates.where(coverage_table.candidate_updates.gt(0)))
display(coverage_table)

### One incoming segment, four rows, one time axis

1. **Traversal time:** final reported destination arrival minus upstream departure, in seconds, excluding upstream dwell. Individual completed traversals and their UTC-bin medians are compared with the full-snapshot median **individual traversal** for this exact directed, shape-specific segment. Each dated visit contributes once, not once per publication. Missing or invalid endpoints remain gaps, without speed or schedule fallback.
2. **Spatial delay change:** for one trip instance and adjacent scheduled visits, **current arrival_delay - upstream arrival_delay**. Positive means added delay, negative means recovery. This is not a temporal difference between successive buses.
3. **Current-stop delay:** the current visit's final reported arrival-delay proxy. Valid current visits remain here even if upstream pairing fails. Positive means late; negative means early.
4. **Preceding-stop delay:** the upstream visit's final reported arrival-delay proxy, plotted at **its own upstream arrival time**, not the later current-stop arrival. Valid upstream visits remain here even if the current visit is missing or unusable. Standalone delay rows exclude arrivals before the recording window; upstream boundary evidence is still retained for paired delay changes.

All rows show individual visits plus five-minute medians, not rolling averages over buses. Empty bins stay gaps; zero reference lines are not missing-data imputations. Traversal time is in seconds and delay in minutes. Hover reveals support, traversal endpoint times, and paired endpoint delays. Traversal support is independent of usable delay estimates. The full-snapshot baseline does not change when zooming.

The shared clock does **not** establish simultaneous measurements: rows 1-3 use destination arrivals, while row 4 uses each preceding-stop arrival. Traversal and delay rows can have different usable visits. Longer traversal relative to its segment median is a congestion proxy, not proof of traffic causality: signals and in-segment holding remain included. Box zoom can change both axes in every row; the range selector/slider controls the shared time window without locking vertical zoom. The legend sits below the plot, separate from the title and time controls. Optional bounds below accept timezone-aware timestamps; naive values are interpreted as UTC.

In [ ]:
display_start = None
display_end = None

plot_started = perf_counter()
bin_times = segment_timing.bins.index.tz_convert(display_timezone)
figure = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.055,
    subplot_titles=('Traversal time vs segment median', 'Delay gained / recovered on the incoming segment', 'Delay at the current stop', 'Delay at the preceding stop'),
)

In [ ]:
traversal_hover = traversal_visits.assign(
    departure_local=traversal_visits.upstream_departure_time.dt.tz_convert(display_timezone).dt.strftime('%Y-%m-%d %H:%M:%S %Z'),
)[['trip_instance_key', 'from_stop_sequence', 'stop_sequence', 'departure_local']]
figure.add_trace(go.Scattergl(
    x=traversal_visits.arrival_time.dt.tz_convert(display_timezone).tolist(),
    y=traversal_visits.traversal_seconds,
    mode='markers', name='Individual completed traversal',
    marker=dict(color='royalblue', size=5, opacity=0.45),
    customdata=traversal_hover,
    hovertemplate='Destination arrival: %{x}<br>Trip: %{customdata[0]}<br>Sequences: %{customdata[1]} -> %{customdata[2]}<br>Upstream departure: %{customdata[3]}<br>Traversal: %{y:.1f} s<extra>%{fullData.name}</extra>',
), row=1, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=segment_timing.bins.median_traversal_seconds,
    mode='lines+markers', name=f'Median traversal ({time_bin_size})',
    line=dict(color='royalblue'), connectgaps=False,
    customdata=segment_timing.bins[['traversal_count']],
    hovertemplate='%{x}<br>Median traversal: %{y:.1f} s<br>Traversals: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=1, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=segment_timing.bins.baseline_traversal_seconds,
    mode='lines', name='Full-snapshot median traversal',
    line=dict(color='darkorange', dash='dash'),
    hovertemplate='%{x}<br>Snapshot median traversal: %{y:.1f} s<extra>%{fullData.name}</extra>',
), row=1, col=1);

In [ ]:
paired_hover = paired_visits.assign(
    upstream_minutes=paired_visits.upstream_arrival_delay / 60,
    current_minutes=paired_visits.arrival_delay / 60,
)[['trip_instance_key', 'from_stop_sequence', 'stop_sequence', 'upstream_minutes', 'current_minutes']]
figure.add_trace(go.Scattergl(
    x=paired_visits.arrival_time.dt.tz_convert(display_timezone),
    y=paired_visits.delta_delay_seconds / 60,
    mode='markers', name='Individual trip delay change',
    marker=dict(color='firebrick', size=5, opacity=0.45),
    customdata=paired_hover,
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Sequences: %{customdata[1]} -> %{customdata[2]}<br>Upstream: %{customdata[3]:.2f} min<br>Current: %{customdata[4]:.2f} min<br>Change: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=2, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_delta_delay_seconds / 60,
    mode='lines+markers', name=f'Median delay change ({time_bin_size})',
    line=dict(color='firebrick'), connectgaps=False,
    customdata=eda.delay_bins[['pair_count']],
    hovertemplate='%{x}<br>Median change: %{y:.2f} min<br>Paired visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=2, col=1);

In [ ]:
figure.add_trace(go.Scattergl(
    x=current_visits.arrival_time.dt.tz_convert(display_timezone),
    y=current_visits.arrival_delay / 60,
    mode='markers', name='Individual current-stop delay',
    marker=dict(color='seagreen', size=5, opacity=0.45),
    customdata=current_visits[['trip_instance_key', 'stop_sequence']],
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Stop sequence: %{customdata[1]}<br>Delay: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=3, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_current_delay_seconds / 60,
    mode='lines+markers', name=f'Median current-stop delay ({time_bin_size})',
    line=dict(color='seagreen'), connectgaps=False,
    customdata=eda.delay_bins[['visit_count']],
    hovertemplate='%{x}<br>Median delay: %{y:.2f} min<br>Current visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=3, col=1);

In [ ]:
figure.add_trace(go.Scattergl(
    x=upstream_visits.arrival_time.dt.tz_convert(display_timezone).tolist(),
    y=upstream_visits.arrival_delay / 60,
    mode='markers', name='Individual preceding-stop delay',
    marker=dict(color='mediumpurple', size=5, opacity=0.45),
    customdata=upstream_visits[['trip_instance_key', 'stop_sequence']],
    hovertemplate='%{x}<br>Trip: %{customdata[0]}<br>Stop sequence: %{customdata[1]}<br>Delay: %{y:.2f} min<extra>%{fullData.name}</extra>',
), row=4, col=1)
figure.add_trace(go.Scatter(
    x=bin_times, y=eda.delay_bins.median_upstream_delay_seconds / 60,
    mode='lines+markers', name=f'Median preceding-stop delay ({time_bin_size})',
    line=dict(color='mediumpurple'), connectgaps=False,
    customdata=eda.delay_bins[['upstream_visit_count']],
    hovertemplate='%{x}<br>Median delay: %{y:.2f} min<br>Preceding visits: %{customdata[0]}<extra>%{fullData.name}</extra>',
), row=4, col=1);

In [ ]:
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=2, col=1)
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=3, col=1)
figure.add_hline(y=0, line_dash='dot', line_color='gray', row=4, col=1)
figure.update_yaxes(title_text='Traversal time (s)', rangemode='tozero', row=1, col=1)
figure.update_yaxes(title_text='Delay change (min)', row=2, col=1)
figure.update_yaxes(title_text='Current delay (min)', row=3, col=1)
figure.update_yaxes(title_text='Preceding delay (min)', row=4, col=1)
figure.update_yaxes(fixedrange=False)
view_start = pd.to_datetime(display_start, utc=True) if display_start is not None else segment_timing.bins.index[0]
view_end = pd.to_datetime(display_end, utc=True) if display_end is not None else segment_timing.bins.index[-1] + pd.Timedelta(time_bin_size)
if pd.isna(view_start) or pd.isna(view_end) or view_start >= view_end:
    raise ValueError('Display bounds must be valid timestamps with start before end.')
figure.update_xaxes(range=[view_start.tz_convert(display_timezone), view_end.tz_convert(display_timezone)])
figure.update_xaxes(
    title_text=f'Time ({display_timezone})', row=4, col=1,
    rangeslider=dict(visible=True, thickness=0.06),
    rangeselector=dict(buttons=[
        dict(count=7, label='7d', step='day', stepmode='backward'),
        dict(count=1, label='1m', step='month', stepmode='backward'),
        dict(label='All', step='all'),
    ]),
)
if traversal_visits.empty:
    figure.add_annotation(text='No usable traversals: inspect timing audit', x=0.5, y=0.5, xref='x domain', yref='y domain', showarrow=False)
if paired_visits.empty:
    figure.add_annotation(text='No usable paired delays: inspect audit', x=0.5, y=0.5, xref='x2 domain', yref='y2 domain', showarrow=False)
if current_visits.empty:
    figure.add_annotation(text='No usable current-stop delays: inspect audit', x=0.5, y=0.5, xref='x3 domain', yref='y3 domain', showarrow=False)
if upstream_visits.empty:
    figure.add_annotation(text='No usable preceding-stop delays: inspect audit', x=0.5, y=0.5, xref='x4 domain', yref='y4 domain', showarrow=False)
figure.update_layout(
    title=f"Segment {segment_id}, direction {trip_direction_id}, shape {eda.segment['shape_id']}<br>{eda.segment['from_stop_name']} -> {eda.segment['to_stop_name']}",
    height=1300, hovermode='x unified', hoversubplots='axis',
    legend=dict(orientation='h', x=0, xanchor='left', y=-0.18, yanchor='top'),
    margin=dict(t=140, b=300),
)
plot_seconds = perf_counter() - plot_started
print(f'Figure construction: {plot_seconds:.2f} s (browser rendering is separate)')
figure.write_html(visuals_dir / 'incoming_segment_delay_dynamics.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "incoming_segment_delay_dynamics.html"}')

### Delay change, speed, traversal and dwell: four rows, one time axis

Can a completed segment's elapsed time provide clearer context than noisy instantaneous speeds? Keep the same selected incoming segment and compare:

1. **Arrival-delay change:** destination arrival delay minus upstream arrival delay, unchanged from the first figure.
2. **Bus speed:** time-binned mean instantaneous probe speed and full-snapshot median individual probe speed, in km/h, retained here for comparison with traversal time.
3. **Traversal time:** destination arrival minus **upstream departure**, in seconds. Upstream dwell is already excluded; do not subtract it again.
4. **Upstream-stop dwell:** upstream departure minus upstream arrival, in seconds, including reported zero dwell. This occurs before traversal and can contribute to the downstream arrival-delay change; destination dwell occurs afterward and is not used here.

**No overlapping elapsed time:** dwell is `[upstream arrival, upstream departure)`; traversal is `[upstream departure, destination arrival)`. They meet only at upstream departure. Their sum is arrival-to-arrival elapsed time, matching the interval underlying the first row. Arrival-delay change is this elapsed time **minus the scheduled arrival-to-arrival interval**, not the raw sum of durations or their deviations from separate medians. These rows provide context rather than proof of causality.

Duration rows show individual visits, time-bin medians, and dashed **full-snapshot medians for this exact directed, shape-specific segment**. Each usable visit contributes once to a duration baseline; the speed baseline still weights individual probes. Changing the display window never recalculates baselines. Timing support is independent of delay support. Both durations require upstream departure, but missing upstream arrival need not hide a usable traversal; destination departure is not required. Empty bins remain gaps, not zeros. Inspect the independent timing exclusions and support below.

Rows 1, 3 and 4 are anchored at **destination arrival** to compare the same dated segment visit; upstream dwell occurs before that plotted anchor. Speed keeps its publication clock. Hover exposes upstream arrival/departure and destination arrival times, dated trip/sequence identity and bin support. A common clock does not imply simultaneous observations or that speed probes belong to the same buses.

**Measurement limits:** these are latest reported feed-estimate proxies, not measured door-open durations. Missing final endpoints are not replaced with older predictions or scheduled times. Traversal can include traffic signals and in-segment holding, so it is not pure road congestion. Longer segments naturally take more time: this plot does not establish a causal or length-dependent effect. Full-snapshot medians and final estimates are retrospective, not point-in-time forecasting features.

In [ ]:
timing_summary = pd.DataFrame({
    'metric': ['Traversal', 'Upstream-stop dwell'],
    'usable_visits': [segment_timing.bins.traversal_count.sum(), segment_timing.bins.dwell_count.sum()],
    'full_snapshot_median_seconds': [
        segment_timing.bins.baseline_traversal_seconds.iloc[0],
        segment_timing.bins.baseline_dwell_seconds.iloc[0],
    ],
})
display(timing_summary)
display(segment_timing.audit)

In [ ]:
timing_figure = make_segment_timing_figure(
    eda, segment_timing, display_timezone=display_timezone,
    display_start=display_start, display_end=display_end,
)
timing_figure.write_html(visuals_dir / 'incoming_segment_timing_comparison.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "incoming_segment_timing_comparison.html"}')

### What this EDA can tell us

- Do completed traversals and paired delays overlap often enough in time to support further study? Are the patterns consistent across well-supported days rather than driven by one bus or sparse bins?
- Do above-median traversal times coincide with delay gains, or do buses remain late without adding delay on this segment? In the first figure, row 2 separates accumulated lateness from changes on the incoming segment; rows 3-4 show the current and preceding stop's total state-like delay proxies at their respective passage times. The separate comparison figure retains instantaneous speed as context.
- How much evidence is missing because of historical GTFS mismatch, missing trip starts, or incomplete endpoints? A missing bin is unknown, not uncongested.
- Does departure-to-arrival traversal time depart from its segment median when arrival-delay change grows? Is upstream-stop dwell unusual too? Both contribute to arrival-to-arrival elapsed time, while the scheduled interval determines how much delay is gained or recovered.

**Limits:** arrival-to-arrival changes include upstream dwell, holding, and schedule slack, so they are not pure congestion measurements. Full-snapshot traversal/speed baselines and final reported delays can use information unavailable at an earlier prediction time. This is retrospective EDA, not a leakage-free backtest or proof of causality, forecast skill, or state-space parameter identifiability. Same-trip upstream/current arrival delays also refer to different passage times, not simultaneous stop states at t.

The modelling section below takes the first step: fit a same-journey stop-to-stop equation on individual eligible visits and evaluate on later held-out days, with training-only duration baselines. It does not define simultaneous stop states, a fixed time grid, external actual-arrival measurements or pre-arrival feature availability. Those remain requirements for a future operational state-space forecast.

### Follow individual journeys across all their stops

The earlier figure holds one segment fixed and compares successive buses. Here, hold a **dated journey** fixed (`trip_id` plus its reliable start descriptor) and follow its entire scheduled stop sequence on the selected `route_id` and `trip_direction_id`, not just `segment_id`. A physical bus can operate many journeys. Express and short-turn journeys keep their own scheduled stop patterns.

First rank every eligible journey by its **maximum signed final arrival delay**, then inspect the histogram of these maxima. Each journey contributes exactly once; repeated prediction updates are not extra journeys. Negative maxima mean the journey was early at every usable stop. Missing/blank starts, historical GTFS mismatches, and unusable final predictions are audited, not guessed or replaced by older predictions.

**Incomplete journeys remain included.** The catalogue reports usable/scheduled stops and coverage: an observed maximum can understate the maximum over a fully observed journey. Stops with no usable observation remain visible as gaps. These maxima and full-snapshot traversal-time/upstream-dwell medians are retrospective descriptions, not point-in-time forecasts or evidence that congestion caused a delay.

In [ ]:
journey_prepare_started = perf_counter()
journey_eda = prepare_trip_dynamics_eda(
    dataset.vehicle_updates, dataset.trip_updates, stops, stop_times, trips,
    route_id=route_id, trip_direction_id=trip_direction_id,
)
journey_catalogue = journey_eda.journeys
print(f'Eligible dated journeys: {len(journey_catalogue):,}; preparation: {perf_counter() - journey_prepare_started:.2f} s')
print(f'Incomplete stop-delay coverage: {journey_catalogue.incomplete.sum():,} journeys')
display(journey_eda.audit)
display(journey_catalogue)

#### Choose a maximum delay or a reference journey

The current controls select the **explicit dated reference journey below**, overriding the numeric/percentile target. Set `reference_trip_instance_key = None` to use the **25th percentile of observed journey maxima** instead. Set `target_max_delay_minutes` to a number to override that percentile. Choose `k` below (currently **5**). Candidates are ranked by **`abs(journey_max_delay - target)`**, not merely by the largest delay. There is **no fixed +/- tolerance cutoff**. Equally close matches favor higher stop coverage, then a stable trip-instance key. If fewer than k journeys exist, all available journeys are returned with a warning.

Choose `reference_trip_instance_key` as a **full dated key from `journey_catalogue`**, not a bare GTFS `trip_id` that repeats across dates. Its maximum delay becomes the target, overriding the numeric/percentile controls. That reference journey is **always plotted first**, followed by the **k - 1 other journeys** closest to its maximum delay, with no duplicates. Thus k includes the reference itself; k = 1 plots only that journey. `None` enables the numeric/percentile selection described above. An unknown reference raises an error rather than selecting a different journey.

Rerun the selection, histogram, and detailed plots after changing these controls; the cached data and journey preparation need not be rerun. Black marks on the histogram identify the selected maxima; near-identical marks can overlap, so use the selection table for exact values.

In [ ]:
reference_trip_instance_key = None
reference_trip_instance_key = "2930135:15557228:15569737|2026-10-26 11:04:00"
journey_max_delay_quantile = 0.25
target_max_delay_minutes = None
k = 5

In [ ]:
selected_journeys = select_trip_dynamics_examples(
    journey_catalogue, target_max_delay_minutes=target_max_delay_minutes,
    max_delay_quantile=journey_max_delay_quantile, k=k,
    reference_trip_instance_key=reference_trip_instance_key,
)
resolved_target_minutes = selected_journeys.target_max_delay_minutes.iloc[0]
if reference_trip_instance_key is not None:
    print(f'Reference journey (first figure): {reference_trip_instance_key}')
print(f'Target maximum delay: {resolved_target_minutes:.3f} min; selected {len(selected_journeys)} of {len(journey_catalogue):,} journeys')
display(selected_journeys[[
    'trip_instance_key', 'shape_id', 'vehicle_id', 'vehicle_identity_status',
    'max_delay_minutes', 'absolute_difference_minutes', 'usable_stop_count',
    'scheduled_stop_count', 'stop_coverage', 'incomplete',
]])

In [ ]:
journey_maximum_histogram = go.Figure(go.Histogram(
    x=journey_catalogue.max_delay_minutes, nbinsx=40, name='Journey maxima',
    marker_color='royalblue',
    hovertemplate='Maximum delay: %{x:.2f} min<br>Journeys in bin: %{y}<extra></extra>',
))
journey_maximum_histogram.add_vline(
    x=resolved_target_minutes, line_color='firebrick', line_dash='dash',
    annotation_text=f'Target: {resolved_target_minutes:.3f} min', annotation_position='top right',
)
journey_maximum_histogram.add_trace(go.Scatter(
    x=selected_journeys.max_delay_minutes, y=[0] * len(selected_journeys),
    mode='markers', name='Selected journey maxima', cliponaxis=False,
    marker=dict(color='black', symbol='line-ns', size=18, line=dict(color='black', width=2)),
    customdata=selected_journeys[['trip_instance_key', 'absolute_difference_minutes']],
    hovertemplate='Journey: %{customdata[0]}<br>Maximum: %{x:.3f} min<br>Absolute distance to target: %{customdata[1]:.3f} min<extra></extra>',
))
journey_maximum_histogram.update_layout(
    title=f'{route_id}, direction {trip_direction_id}: maximum arrival delay per dated journey',
    xaxis_title='Observed maximum signed arrival delay (min)',
    yaxis=dict(title='Number of journeys', rangemode='nonnegative'),
    height=500, bargap=0.05, margin=dict(t=90, b=110),
    legend=dict(orientation='h', y=-0.25),
)
journey_maximum_histogram.write_html(visuals_dir / 'journey_maximum_delay_histogram.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "journey_maximum_delay_histogram.html"}')

#### How does delay evolve within each selected journey?

Each figure shares a **scheduled-stop axis in travel order** (not elapsed time or distance):

1. **Arrival delay:** minutes late/early at each stop. A high, flat line means lateness persists without further accumulation; rising/falling sections mean growing/recovering delay.
2. **Delay gained/recovered:** signed current-minus-preceding arrival delay on each incoming segment. Red adds delay; green recovers it. Missing intermediate stops are never bridged, and the origin has no incoming segment.
3. **Traversal-time context:** this dated journey's final reported destination arrival minus its own immediately preceding stop's final departure, in seconds, versus the full-snapshot median individual traversal for each exact directed, shape-specific segment. Upstream dwell is excluded. Longer segments naturally take more time, so compare each value with its own segment median. Hover includes departure/arrival endpoints, validity, and baseline traversal counts. Timing availability is independent of arrival-delay or vehicle-probe availability; a reliable dated trip/sequence match is required, not a unique physical-bus ID. The origin and missing, skipped, unfinished or nonpositive traversals stay gaps; missing intermediate stops are never bridged, and older predictions or speeds are never substituted. Baselines include all usable snapshot traversals, not only the selected journeys.
4. **Upstream-stop dwell:** for each incoming segment, upstream departure minus upstream arrival, in seconds, alongside the full-snapshot **median** of usable upstream dwell observations for that exact directed, shape-specific segment. Each dated visit contributes once, including reported zero dwell; repeated publications and the chosen journey subset do not reweight it. Hover identifies the upstream stop, arrival/departure endpoints, validity and baseline support. The origin and unavailable, skipped, unfinished or negative dwell values stay gaps. Dwell and traversal share only the upstream-departure boundary: their elapsed times do not overlap. The dwell value is aligned with the downstream stop whose arrival-delay change it can affect, not with the destination's later dwell.

The largest positive bar identifies where the most delay was gained, which need not be the stop with the largest total delay. Follow the line downstream to see persistence, additional accumulation, or recovery. Delay differences also include upstream dwell, holding, schedule slack, and prediction revisions. Traversal time includes signals and in-segment holding; a longer traversal alongside growing delay does not establish a causal road-congestion effect.

In [ ]:
for journey_number, trip_key in enumerate(selected_journeys.trip_instance_key, start=1):
    journey_figure = make_trip_dynamics_figure(
        journey_eda, trip_key, display_timezone=display_timezone,
    )
    journey_figure.write_html(visuals_dir / f'journey_dynamics_{journey_number:02d}.html', full_html=True, include_plotlyjs=True)
    print(f'Saved: {visuals_dir / f"journey_dynamics_{journey_number:02d}.html"}')
    del journey_figure

## Modelling: same-journey delay dynamics for segment 4

For this incoming segment, fit the proposed equation by ordinary least squares:

$$\widehat D_{\mathrm{current}}(t) = \alpha_d D_{\mathrm{upstream}}(t-1) + \alpha_t[T(t)-B_T(h)] + \alpha_w[W(t)-B_W(h)] - C.$$

All quantities are in **seconds**; the three alpha coefficients are dimensionless and `C` is a signed fitted constant, not constrained to be positive. `T` is destination arrival minus upstream departure; `W` is upstream departure minus upstream arrival. They do not overlap. `h` is the **Vancouver local hour of that same journey's upstream arrival**. `B_T(h)` and `B_W(h)` are medians of eligible **training visits** in that hour. They are fixed after training and repeat daily; this first model does not add weekday or seasonal effects. An hour absent from training uses global training medians, explicitly labelled with zero hourly support and warned about if used in holdout.

### Model-specific preprocessing and chronological split

Require the intersection of valid paired arrival delays, traversal and upstream dwell, preserving reported zero dwell and exact dated stop-sequence matches. Inspect the model audit: EDA rows usable for only one metric cannot enter the fit. Reserve the **last seven observed local destination-arrival days** (not seven randomly selected days or necessarily consecutive calendar days). Exclude training visits with final endpoint publications at/after the holdout boundary and training visits from any held-out journey. No full-snapshot EDA baseline, selected journey subset, display window or five-minute aggregate is used to train.

The fit uses completed durations and **latest reported** delays. `t-1` indexes the same journey's preceding stop; there is no one-bin temporal lag or simultaneous stop-state matrix. Since `T` itself includes the destination arrival timestamp underlying the delay target, this is a **retrospective explanatory fit**, not a pre-arrival forecasting backtest. Train/holdout separation prevents fitting baselines or coefficients on held-out outcomes, but does not make these inputs available before arrival.

In [6]:
model_holdout_days = 7
if (route_id, trip_direction_id, segment_id) != ('97-KEL', 0, 4):
    raise ValueError('This modelling experiment is restricted to Route 97, direction 0, segment 4.')

model_result = fit_stop_dynamics_model(
    eda, segment_timing, holdout_days=model_holdout_days,
    baseline_timezone=display_timezone,
)
print(f'First held-out arrival day: {model_result.holdout_start}')
display(model_result.audit)
display(model_result.observations.groupby('split').agg(
    visits=('arrival_delay', 'size'), observed_days=('arrival_day', 'nunique'),
    first_arrival=('arrival_time', 'min'), last_arrival=('arrival_time', 'max'),
))

/tmp/ipykernel_75840/4205805030.py:5: UserWarning: Some model visits were excluded; inspect model audit.
  model_result = fit_stop_dynamics_model(


First held-out arrival day: 2026-11-04 00:00:00-08:00


,reason,count
0,included,2195
1,delay_current_unfinished_visit,9
2,delay_nonincreasing_arrival_times,2
3,delay_upstream_missing_visit,1
4,delay_upstream_skipped_visit,1
5,delay_current_cancelled_trip,1


,visits,observed_days,first_arrival,last_arrival
split,,,,
holdout,349,7,2026-11-04 13:27:55+00:00,2026-11-10 22:07:39+00:00
train,1846,38,2026-09-12 14:28:05+00:00,2026-11-04 07:17:19+00:00


### Fitted equation and training-only hourly reference durations

Read the equation together with the baseline table; its intercept depends on the chosen centring. A negative `C` adds delay because the equation subtracts `C`. Coefficients are unconstrained: a negative slope or upstream-delay coefficient outside [0, 1] is allowed and should prompt interpretation, not be silently clipped. Sparse hourly support, correlated predictors and feed prediction revisions limit mechanistic interpretation. These are estimated coefficients, not independently established physical parameters.

In [7]:
coefficients = model_result.coefficients
fitted_equation = (
    f"D_current = {coefficients['alpha_d']:.6f} * D_upstream "
    f"+ {coefficients['alpha_t']:.6f} * (T - B_T(h)) "
    f"+ {coefficients['alpha_w']:.6f} * (W - B_W(h)) "
    f"- ({coefficients['constant_seconds']:.6f})"
)
print(fitted_equation)
print('Delays, durations, hourly baselines and the signed constant are in seconds.')
display(coefficients.to_frame())
display(model_result.hourly_baselines)

D_current = 0.995836 * D_upstream + 0.900823 * (T - B_T(h)) + 0.400842 * (W - B_W(h)) - (-14.996222)
Delays, durations, hourly baselines and the signed constant are in seconds.


,value
alpha_d,0.995836
alpha_t,0.900823
alpha_w,0.400842
constant_seconds,-14.996222


,baseline_traversal_seconds,baseline_dwell_seconds,training_count,baseline_source
baseline_hour,,,,
0,334.0,0.0,18,hourly_training_median
1,420.0,16.0,0,global_training_median
2,420.0,16.0,0,global_training_median
3,420.0,16.0,0,global_training_median
4,420.0,16.0,0,global_training_median
5,385.0,0.0,5,hourly_training_median
6,383.0,0.0,65,hourly_training_median
7,488.5,1.0,116,hourly_training_median
8,600.0,20.0,114,hourly_training_median


### Held-out evaluation on the same eligible visits

Compare the equation with **upstream-delay persistence** (`D_current = D_upstream`) and **mean delay change** (`D_current = D_upstream + mean_training(D_current - D_upstream)`). All three are evaluated against the same original signed arrival-delay targets, without removing extreme delays or clipping early arrivals. MAE and RMSE are in seconds; R-squared is dimensionless. Training errors describe fit, not generalization. A lower held-out error than persistence measures retrospective explanatory improvement only.

Matching time-series figures show individual training and held-out observations and estimates at destination arrival, followed by residuals (`reported - fitted`): `segment_4_dynamics_training.html` and `segment_4_dynamics_holdout.html`. The actual-versus-predicted figure below compares reported downstream arrival delay with the fitted equation, distinguishing training from holdout. Remaining residual structure may reflect schedule slack, hourly baseline limitations, holding or feed inconsistency; it is not automatically unexplained road congestion. Exports preserve the equation, coefficients, hourly support/baselines, split metadata, metrics, eligibility audit and per-visit predictions for this selected dataset; rerunning overwrites these files.

In [8]:
display(model_result.metrics)
holdout_scores = model_result.metrics.query("split == 'holdout'").set_index('model')
equation_mae = holdout_scores.loc['equation', 'mae_seconds']
persistence_mae = holdout_scores.loc['persistence', 'mae_seconds']
print(f'Held-out MAE: equation {equation_mae:.2f} s; upstream persistence {persistence_mae:.2f} s')

coefficients.to_csv(visuals_dir / 'segment_4_dynamics_coefficients.csv')
model_result.hourly_baselines.to_csv(visuals_dir / 'segment_4_dynamics_hourly_baselines.csv')
model_result.metrics.to_csv(visuals_dir / 'segment_4_dynamics_metrics.csv', index=False)
model_result.audit.to_csv(visuals_dir / 'segment_4_dynamics_audit.csv', index=False)
model_result.observations.to_csv(visuals_dir / 'segment_4_dynamics_predictions.csv', index=False)
model_metadata = pd.Series({
    'dataset_id': backtest_dataset_id, 'route_id': route_id,
    **model_result.segment.to_dict(),
    'holdout_observed_days': model_holdout_days,
    'holdout_start': model_result.holdout_start.isoformat(),
    'baseline_timezone': model_result.baseline_timezone,
    'equation_seconds': fitted_equation,
})
model_metadata.to_csv(visuals_dir / 'segment_4_dynamics_metadata.csv')
print(f'Saved fitted equation and evaluation CSV tables under: {visuals_dir}')

,split,model,n,mae_seconds,rmse_seconds,r2,r2_undefined_reason
0,train,equation,1846,39.076447,53.427547,0.926414,
1,train,persistence,1846,75.846696,132.462953,0.547671,
2,train,mean_change,1846,80.452589,127.371195,0.581777,
3,holdout,equation,349,43.095418,70.616168,0.926591,
4,holdout,persistence,349,61.515759,115.958186,0.802056,
5,holdout,mean_change,349,74.136991,117.993267,0.795047,


Held-out MAE: equation 43.10 s; upstream persistence 61.52 s


Saved fitted equation and evaluation CSV tables under: /home/abdul/Projects/transit-forecasting/notebooks/visuals/dynamics_model_notebook


In [4]:
model_figure = make_stop_dynamics_model_figure(
    model_result, split='holdout', display_timezone=display_timezone,
)
model_figure.write_html(visuals_dir / 'segment_4_dynamics_holdout.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "segment_4_dynamics_holdout.html"}')

Saved: /home/abdul/Projects/transit-forecasting/notebooks/visuals/dynamics_model_notebook/segment_4_dynamics_holdout.html


### Training time series and actual versus predicted

The training plot mirrors the holdout plot but uses only the visits that fitted the equation. It describes in-sample fit, not forecast skill. The scatter plot places **actual (final reported) arrival delay on x** and **fitted arrival delay on y**, both in minutes, with equal axis scales and a dashed **1:1 line**. Points above the line overpredict delay; points below underpredict. Training and holdout remain separate legend groups; toggle either to inspect it alone. Here, actual is a reported feed proxy, not independently measured traffic congestion.

In [5]:
training_figure = make_stop_dynamics_model_figure(
    model_result, split='train', display_timezone=display_timezone,
)
training_figure.write_html(visuals_dir / 'segment_4_dynamics_training.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "segment_4_dynamics_training.html"}')

Saved: /home/abdul/Projects/transit-forecasting/notebooks/visuals/dynamics_model_notebook/segment_4_dynamics_training.html


In [6]:
actual_predicted_figure = go.Figure()
for split, label, color in (('train', 'Training', 'royalblue'), ('holdout', 'Holdout', 'darkorange')):
    visits = model_result.observations.loc[model_result.observations['split'].eq(split)]
    actual_predicted_figure.add_trace(go.Scattergl(
        x=visits.arrival_delay / 60, y=visits.predicted_delay_seconds / 60,
        mode='markers', name=f'{label} ({len(visits):,} visits)',
        marker=dict(color=color, size=6, opacity=0.55),
        customdata=visits[['trip_instance_key', 'stop_sequence']],
        hovertemplate='Journey: %{customdata[0]}<br>Stop sequence: %{customdata[1]}<br>Actual reported: %{x:.2f} min<br>Predicted: %{y:.2f} min<extra>%{fullData.name}</extra>',
    ))
delay_values = model_result.observations[['arrival_delay', 'predicted_delay_seconds']] / 60
low, high = delay_values.min().min(), delay_values.max().max()
padding = max((high - low) * 0.05, 0.1)
axis_bounds = [low - padding, high + padding]
actual_predicted_figure.add_trace(go.Scatter(
    x=axis_bounds, y=axis_bounds, mode='lines', name='1:1 (perfect agreement)',
    line=dict(color='black', dash='dash'), hoverinfo='skip',
))
actual_predicted_figure.update_layout(
    title=f'Segment {segment_id}: retrospective dynamics actual vs predicted',
    xaxis=dict(title='Actual final reported arrival delay (min)', range=axis_bounds, constrain='domain'),
    yaxis=dict(title='Predicted arrival delay (min)', range=axis_bounds, scaleanchor='x', scaleratio=1),
    height=800, margin=dict(t=90, b=140),
    legend=dict(orientation='h', x=0, y=-0.18, yanchor='top'),
)
actual_predicted_figure.write_html(visuals_dir / 'segment_4_dynamics_actual_vs_predicted.html', full_html=True, include_plotlyjs=True)
print(f'Saved: {visuals_dir / "segment_4_dynamics_actual_vs_predicted.html"}')

Saved: /home/abdul/Projects/transit-forecasting/notebooks/visuals/dynamics_model_notebook/segment_4_dynamics_actual_vs_predicted.html


### Interpretation and next step

This equation estimates how the final reported downstream delay relates to the same journey's upstream delay and non-overlapping elapsed durations on **segment 4 only**. If the feed is internally consistent, arrival-delay change equals elapsed arrival-to-arrival time minus the scheduled arrival interval, so slopes near one can largely recover timetable arithmetic. Hourly median baselines are typical durations, not scheduled intervals or uncongested free-flow times. The fitted constant and residuals can absorb schedule slack and reporting differences.

Do not treat the holdout score as evidence that congestion can be forecast before traversal finishes. That requires point-in-time upstream observations and forecasts or externally supplied future traversal/dwell measurements, followed by a separate availability-aware evaluation. Do not interpret `alpha_d` as fixed-grid temporal stability or extrapolate these coefficients to other segments, directions or GTFS snapshots.